# Segmentation — train, predict, evaluate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/Segmentation/Segmentation_demo.ipynb)

**Run all the cells.** This notebook trains a multiclass UNet on five public pelvis CT cases,
predicts the segmentation, and scores it with Dice — the complete KonfAI loop, driven only by the
three YAML files sitting next to this notebook.

Nothing is gated behind a flag: every cell does its work. Expect roughly **7 minutes on a GPU**
(longer on CPU; on Colab pick *Runtime > Change runtime type > GPU*).

In [ ]:
# Setup: find KonfAI (cloning it on Colab), install what is missing, load the notebook helpers.
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples").is_dir())
sys.path.insert(0, str(REPO_DIR / "examples"))

from konfai_demo import latest_checkpoint, read, run, setup, show

EXAMPLE_DIR, DATASET_DIR, DEVICE = setup(REPO_DIR, "Segmentation", ("konfai", f"{REPO_DIR}[imaging]"), "huggingface_hub", "matplotlib")

## 1. The data

Five pelvis CT cases with a 41-label reference segmentation (~114 MB), one folder per case:

```text
Dataset/1PC006/CT.mha
Dataset/1PC006/SEG.mha
```

`CT` is the model input, `SEG` the label map it must reproduce.

In [ ]:
import shutil

from huggingface_hub import snapshot_download

if not any(DATASET_DIR.glob("*/CT.mha")):
    snapshot_download("VBoussot/konfai-demo", repo_type="dataset", allow_patterns="Segmentation/**", local_dir=str(DATASET_DIR))
    for case in (DATASET_DIR / "Segmentation").iterdir():
        shutil.move(str(case), DATASET_DIR / case.name)
    shutil.rmtree(DATASET_DIR / "Segmentation")
    shutil.rmtree(DATASET_DIR / ".cache", ignore_errors=True)

CASES = sorted(path.name for path in DATASET_DIR.iterdir() if path.is_dir())
print(len(CASES), "cases:", ", ".join(CASES))

## 2. Look at one case

The CT alone, then with its reference labels drawn on top.

In [ ]:
import numpy as np

case = DATASET_DIR / CASES[0]
ct, seg = read(case / "CT.mha"), read(case / "SEG.mha")
LABEL_MAX = int(seg.max())
print(case.name, "| shape:", ct.shape, "| labels:", np.unique(seg))

BEST = int(np.argmax((seg > 0).sum(axis=(1, 2))))  # the slice with the most annotated voxels
show([(f"CT — slice {BEST}", ct[BEST], "gray"), ("reference SEG", ct[BEST], "gray", seg[BEST])], LABEL_MAX)

## 3. Train, predict, evaluate

Three commands, one YAML file each — this is the whole framework:

| Command | Config | What it does |
|---|---|---|
| `konfai TRAIN` | `Config.yml` | trains the UNet declared in `UNet.yml`, writes `Checkpoints/SEG_BASELINE/` |
| `konfai PREDICTION` | `Prediction.yml` | reassembles a segmentation per case into `Predictions/SEG_BASELINE/` |
| `konfai EVALUATION` | `Evaluation.yml` | scores prediction vs reference into `Evaluations/SEG_BASELINE/` |

The training is deliberately short so the notebook finishes; the accuracy is a starting point, not a
result. Raise `epochs` in `Config.yml` for a real run.

In [ ]:
run("konfai", "TRAIN", "-y", *DEVICE, "--config", "Config.yml")

run("konfai", "PREDICTION", "-y", *DEVICE, "--config", "Prediction.yml", "--models", latest_checkpoint("SEG_BASELINE"))
run("konfai", "EVALUATION", "-y", "--config", "Evaluation.yml")

## 4. The result

`Evaluations/SEG_BASELINE/Metric_TRAIN.json` holds a Dice per case plus the aggregates. Below, the
reference and the prediction side by side on the same slice.

In [ ]:
import json

metrics = json.loads((EXAMPLE_DIR / "Evaluations" / "SEG_BASELINE" / "Metric_TRAIN.json").read_text())
for name, values in metrics["aggregates"].items():
    print(f"{name:24s} mean {values['mean']:.3f}   min {values['min']:.3f}   max {values['max']:.3f}")

prediction = read(EXAMPLE_DIR / "Predictions" / "SEG_BASELINE" / "Dataset" / case.name / "PRED.mha")
show([
    (f"CT — slice {BEST}", ct[BEST], "gray"),
    ("reference", ct[BEST], "gray", seg[BEST]),
    ("prediction", ct[BEST], "gray", prediction[BEST]),
], LABEL_MAX)

## What to change next

- **more training** — raise `epochs` in `Config.yml`; checkpoints land in `Checkpoints/SEG_BASELINE/`
  and TensorBoard logs in `Statistics/SEG_BASELINE/`.
- **your own data** — point `dataset_filenames` at a folder laid out the same way, and set `nb_class`.
- **another model** — `Config.yml` reads `classpath: UNet.yml`; switch it to `classpath: Model:UNet`
  for the identical network written in Python (`Model.py`), or to any `konfai.models` entry.

`README.md` in this folder covers the config keys in detail.